# Notebook 05 — Final Load Preparation for Tableau

**Objective:** Produce the final, Tableau-optimised output files from the cleaned dataset. This includes KPI summary tables, aggregated views, and the master export with all engineered features. All outputs are committed to `data/processed/`.

In [ ]:
import pandas as pd

import numpy as np

import warnings

warnings.filterwarnings('ignore')

PROCESSED_DIR = '/content/startups_cleaned.csv'

df = pd.read_csv('/content/startups_cleaned.csv', low_memory=False)

print(f'Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns')

In [ ]:

tableau_cols = [

    'name', 'status', 'country_code', 'market', 'primary_category',

    'founding_decade', 'founded_year', 'founded_at', 'first_funding_at', 'last_funding_at',

    'funding_rounds', 'funding_total_usd', 'avg_funding_per_round',

    'seed', 'venture', 'angel', 'round_A', 'round_B', 'round_C',

    'days_to_first_funding', 'funding_duration_days',

    'is_closed', 'is_usa', 'reached_series_a'

]

df_tableau = df[[c for c in tableau_cols if c in df.columns]].copy()

rename_map = {

    'name': 'Startup Name',

    'status': 'Status',

    'country_code': 'Country',

    'market': 'Market',

    'primary_category': 'Primary Category',

    'founding_decade': 'Founding Decade',

    'founded_year': 'Founded Year',

    'founded_at': 'Founded Date',

    'first_funding_at': 'First Funding Date',

    'last_funding_at': 'Last Funding Date',

    'funding_rounds': 'Funding Rounds',

    'funding_total_usd': 'Total Funding USD',

    'avg_funding_per_round': 'Avg Funding Per Round USD',

    'seed': 'Seed Amount USD',

    'venture': 'Venture Amount USD',

    'angel': 'Angel Amount USD',

    'round_A': 'Series A USD',

    'round_B': 'Series B USD',

    'round_C': 'Series C USD',

    'days_to_first_funding': 'Days to First Funding',

    'funding_duration_days': 'Funding Duration Days',

    'is_closed': 'Closed (0/1)',

    'is_usa': 'USA Based (0/1)',

    'reached_series_a': 'Reached Series A (0/1)'

}

df_tableau.rename(columns=rename_map, inplace=True)

out_path = f'{PROCESSED_DIR}tableau_master.csv'

df_tableau.to_csv(out_path, index=False)

print(f'Master Tableau dataset exported: {out_path}')

print(f'Shape: {df_tableau.shape[0]:,} rows x {df_tableau.shape[1]} columns')

df_tableau.head(3)

In [ ]:
total = len(df)

kpi_data = {

    'KPI': [

        'Total Startups Analysed',

        'Overall Failure Rate (%)',

        'Operating Rate (%)',

        'Acquisition Rate (%)',

        'IPO Rate (%)',

        'Median Funding — Closed ($)',

        'Median Funding — Operating ($)',

        'Funding Gap (Operating vs Closed) ($)',

        'Avg Funding Rounds — Closed',

        'Avg Funding Rounds — Operating',

        'Avg Funding Rounds — Acquired',

        'Series A Post-Funding Failure Rate (%)',

        'Pre-Series A Failure Rate (%)',

        'USA Startup Failure Rate (%)',

        'Non-USA Startup Failure Rate (%)'

    ],

    'Value': [

        total,

        round((df['status'] == 'closed').sum() / total * 100, 2),

        round((df['status'] == 'operating').sum() / total * 100, 2),

        round((df['status'] == 'acquired').sum() / total * 100, 2),

        round((df['status'] == 'ipo').sum() / total * 100, 2),

        int(df[df['status'] == 'closed']['funding_total_usd'].median()),

        int(df[df['status'] == 'operating']['funding_total_usd'].median()),

        int(df[df['status'] == 'operating']['funding_total_usd'].median() -

            df[df['status'] == 'closed']['funding_total_usd'].median()),

        round(df[df['status'] == 'closed']['funding_rounds'].mean(), 2),

        round(df[df['status'] == 'operating']['funding_rounds'].mean(), 2),

        round(df[df['status'] == 'acquired']['funding_rounds'].mean(), 2),

        round(df[df['reached_series_a'] == 1]['is_closed'].mean() * 100, 2),

        round(df[df['reached_series_a'] == 0]['is_closed'].mean() * 100, 2),

        round(df[df['country_code'] == 'USA']['is_closed'].mean() * 100, 2),

        round(df[df['country_code'] != 'USA']['is_closed'].mean() * 100, 2)

    ]

}

df_kpi = pd.DataFrame(kpi_data)

kpi_path = f'{PROCESSED_DIR}kpi_summary.csv'

df_kpi.to_csv(kpi_path, index=False)

print(f'KPI summary exported: {kpi_path}')

print(df_kpi.to_string(index=False))

In [ ]:
country_agg = df.groupby('country_code').agg(

    Total_Startups=('is_closed', 'count'),

    Closed=('is_closed', 'sum'),

    Avg_Funding_USD=('funding_total_usd', 'mean'),

    Median_Funding_USD=('funding_total_usd', 'median'),

    Avg_Rounds=('funding_rounds', 'mean')

).reset_index()

country_agg['Failure_Rate_Pct'] = (country_agg['Closed'] / country_agg['Total_Startups'] * 100).round(2)

country_agg = country_agg[country_agg['Total_Startups'] >= 30].sort_values('Failure_Rate_Pct', ascending=False)

country_path = f'{PROCESSED_DIR}country_level_summary.csv'

country_agg.to_csv(country_path, index=False)

print(f'Country summary exported: {country_path}')

print(country_agg.head(15).to_string(index=False))

In [ ]:
sector_agg = df.groupby('market').agg(

    Total_Startups=('is_closed', 'count'),

    Closed=('is_closed', 'sum'),

    Avg_Funding_USD=('funding_total_usd', 'mean'),

    Avg_Rounds=('funding_rounds', 'mean'),

    Series_A_Rate=('reached_series_a', 'mean')

).reset_index()

sector_agg['Failure_Rate_Pct'] = (sector_agg['Closed'] / sector_agg['Total_Startups'] * 100).round(2)

sector_agg['Series_A_Rate_Pct'] = (sector_agg['Series_A_Rate'] * 100).round(2)

sector_agg = sector_agg[sector_agg['Total_Startups'] >= 50].sort_values('Failure_Rate_Pct', ascending=False)

sector_path = f'{PROCESSED_DIR}sector_level_summary.csv'

sector_agg.to_csv(sector_path, index=False)

print(f'Sector summary exported: {sector_path}')

print(sector_agg.head(20).to_string(index=False))

In [ ]:
yearly_agg = df[df['founded_year'].between(1990, 2022)].groupby('founded_year').agg(

    Total_Startups=('is_closed', 'count'),

    Closed=('is_closed', 'sum'),

    Avg_Funding_USD=('funding_total_usd', 'mean'),

    Avg_Rounds=('funding_rounds', 'mean')

).reset_index()

yearly_agg['Failure_Rate_Pct'] = (yearly_agg['Closed'] / yearly_agg['Total_Startups'] * 100).round(2)

yearly_path = f'{PROCESSED_DIR}yearly_trend_summary.csv'

yearly_agg.to_csv(yearly_path, index=False)

print(f'Yearly trend exported: {yearly_path}')

print(yearly_agg.to_string(index=False))

In [ ]:
import os

outputs = [

    ('tableau_master.csv', 'Main dashboard feed — all startup records'),

    ('kpi_summary.csv', 'Executive KPI table for dashboard header'),

    ('country_level_summary.csv', 'Geographic map view'),

    ('sector_level_summary.csv', 'Sector drill-down view'),

    ('yearly_trend_summary.csv', 'Time trend line chart'),

]

print('=== FINAL LOAD SUMMARY ===')

for fname, desc in outputs:

    path = f'{PROCESSED_DIR}{fname}'

    if os.path.exists(path):

        rows = sum(1 for _ in open(path)) - 1

        size_kb = os.path.getsize(path) / 1024

        print(f'  ✓ {fname:<40} {rows:>6,} rows | {size_kb:>6.1f} KB | {desc}')

    else:

        print(f'  ✗ {fname} — NOT FOUND')

print()

print('All files are ready for Tableau import.')

print('Connect tableau_master.csv as the primary data source.')

print('Use the summary files for pre-aggregated calculated fields and KPI tiles.')